# CauST on real tissue: a DLPFC walkthrough

This notebook runs CauST end to end on the spatialLIBD human DLPFC cohort
through the scanpy-style API (`caust.tl` / `caust.pl`):

1. load five slices from three donors (raw counts + manual layer labels),
2. score every shared gene by **knockout invariance** across donors,
3. select a causal gene set and compare it with highly variable genes on a
   donor the selection never saw.

It uses the dependency-light reference backbone so it runs in a minute on a
laptop; swap `backbone="stagate"` for the graph attention autoencoder once
`pip install "caust[stagate]"` is in place (a few minutes per slice on a GPU).

Requirements: `pip install "caust[viz]"`. The ~35 MB of data is fetched and
checksum-verified on first use.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import caust
from caust.dlpfc import load_dlpfc_cohort

print("caust", caust.__version__)

caust 0.1.0


## 1. Load the cohort

`load_dlpfc_cohort` depth-normalises and log-transforms each slice, aligns the
gene symbols, and keeps a candidate pool of the most variable genes across the
**training** slices — the held-out slice (last in the list) never influences
the pool. Two slices per training donor (Br5292: 151507/151508, Br5595:
151669/151670) so the cross-slice stability term is estimated from four
slices, and the held-out donor Br8100 (151673) is evaluated once.

In [2]:
slices = load_dlpfc_cohort(samples=["151507", "151508", "151669", "151670", "151673"], n_candidates=2000)
for a in slices:
    print(a.obs["sample"].iloc[0], a.obs["donor"].iloc[0], a.shape, "layers:", a.obs["domain"].nunique())

151507 Br5292 (4221, 2000) layers: 7
151508 Br5292 (4381, 2000) layers: 7
151669 Br5595 (3636, 2000) layers: 5
151670 Br5595 (3484, 2000) layers: 5
151673 Br8100 (3611, 2000) layers: 7


## 2. Score genes by knockout invariance

`caust.tl.causal_genes` trains a frozen backbone on each training slice,
silences every gene in silico, measures how far the spatial embedding moves,
and scores each gene by `mean − λ·std` of that effect across donors. Results
land in `.var` and `.uns`, scanpy-style.

In [3]:
train = slices[:-1]
table = caust.tl.causal_genes(train, backbone="simple", lam=1.0)
table.head(15)

,gene,score,mean_delta,std_delta
rank,,,,
1,CLU,0.850894,1.118270,0.267377
2,MTRNR2L12,0.727584,0.831259,0.103674
3,ACTB,0.686852,0.858789,0.171937
4,TMSB10,0.651215,0.752842,0.101627
5,COX6C,0.613809,0.735505,0.121696
6,EEF1A1,0.565322,0.669986,0.104665
7,FTL,0.563555,0.616350,0.052795
8,RPL34,0.561005,0.684667,0.123662
9,RPLP1,0.556636,0.645734,0.089098


In [4]:
ax = caust.pl.ranking(train[0], n=20, markers=["MBP", "PLP1", "NRGN", "SNAP25", "CCK", "CALM1", "ENC1"])

Known cortical layer markers (★) surface with no label supervision.

## 3. Select a gene set and test it on the unseen donor

At the same 25-gene budget, compare the CauST set with the top highly variable
genes. `caust.tl.spatial_domains` trains the backbone on the held-out slice
restricted to each gene set and clusters the embedding (mclust-EEE equivalent);
ARI is against the manual layer annotation.

In [5]:
held = slices[-1]
k = held.obs["domain"].nunique()
caust_genes = caust.tl.select_genes(train[0], n_top_genes=25)

import scipy.sparse as sp
X = held.X.toarray() if sp.issparse(held.X) else np.asarray(held.X)
hvg_genes = held.var_names[np.argsort(-X.var(axis=0))[:25]]

for name, genes in [("CauST", caust_genes), ("HVG", hvg_genes)]:
    aris = []
    for seed in range(3):
        labels = caust.tl.spatial_domains(held, k, backbone="simple", genes=list(genes),
                                          random_state=seed, key_added=f"{name}_domain")
        aris.append(caust.ari(held.obs["domain"], labels))
    print(f"{name:6s} ({len(genes)} genes)  ARI on held-out donor: {np.mean(aris):.3f} +/- {np.std(aris):.3f}")

CauST  (25 genes)  ARI on held-out donor: 0.424 +/- 0.115
HVG    (25 genes)  ARI on held-out donor: 0.272 +/- 0.017


In [6]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
caust.pl.domains(held, "domain", ax=axes[0], title="manual annotation")
caust.pl.domains(held, "HVG_domain", ax=axes[1], title="HVG genes")
caust.pl.domains(held, "CauST_domain", ax=axes[2], title="CauST genes")
plt.tight_layout()

## Where to go next

- `caust.tl.select_genes(adata, n_top_genes, lam=...)` re-scores the stored
  knockout effects with a different λ without re-running knockouts.
- `backbone="stagate"` (or `"graphst"`) uses the GNN backbones; the full
  12-slice transfer benchmark is `caust transfer -c configs/transfer/dlpfc_stagate.yaml`.
- Every config-driven run writes a provenance manifest and can be re-verified
  with `caust verify`.